In [1]:
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt
#plt.rcParams["figure.figsize"] = [4, 4]
torch.manual_seed(0)

from engression import engression
from engression.data.loader import partition_data
import xarray as xr
import numpy as np
import json

import sys
sys.path.append('/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder')
import utils as ut

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [3]:
def data_to_torch(ds, variable):
    temp_data = ds[variable]
    data = temp_data.transpose('time', 'lat', 'lon')
    
    # Now convert to numpy
    data_np = data.values  # Shape: (time, lat, lon)
    
    # Flatten lat and lon together
    time_steps, lat_dim, lon_dim = data_np.shape
    data_np = data_np.reshape(time_steps, lat_dim * lon_dim)
    
    
    data_np = data_np  # Shape: (grid_cell, timestep)
    
    # Finally, convert to torch tensor
    data_tensor = torch.tensor(data_np, dtype=torch.float32)
    print(data_tensor.shape)
    return data_tensor

def remove_nan_columns(tensor: torch.Tensor):
    """
    Removes columns that are entirely NaN from a 2D tensor.
    
    Returns:
        - reduced_tensor: Tensor with NaN-only columns removed
        - column_mask: Boolean mask indicating which columns were kept
    """
    if tensor.dim() != 2:
        raise ValueError("Input must be a 2D tensor")
        
    column_mask = ~torch.isnan(tensor).all(dim=0)
    reduced_tensor = tensor[:, column_mask]
    return reduced_tensor, column_mask

def standardize_numpy(X):
    mean = X.mean(axis=0, keepdims=True)
    print("mean shape:", mean.shape)
    std = X.std(axis=0, keepdims=True)
    print("std shape:", std.shape)
    return (X - mean) / (std), mean, std

In [5]:
# v1 not standardized
#predictors = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/GMT_and_pseudoPCs_EOFs_Z500_5daily_100ensmembers_JJA_not_scaled.nc").pseudo_pcs.values
#print(predictors.shape)
#predictors_test = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/GMT_and_pseudoPCs_EOFs_ETH130014001500_Z500_5daily_100ensmembers_JJA_not_scaled.nc").pseudo_pcs.values
#predictands_pre2 = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/europe_10percent_masked_stacked_TREFHT_JJA.nc")#.TREFHT.values.reshape(32 * 32, *[14307]).T
#print(predictands_pre2)

# v1 standardized
#predictors, _, _ = standardize_numpy(xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/GMT_and_pseudoPCs_EOFs_Z500_5daily_100ensmembers_JJA_not_scaled.nc").pseudo_pcs.values)
#print(predictors.shape)
#predictors_test, _, _ = standardize_numpy(xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/GMT_and_pseudoPCs_EOFs_ETH130014001500_Z500_5daily_100ensmembers_JJA_not_scaled.nc").pseudo_pcs.values)
#predictands_pre2 = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v1_until16102025/europe_10percent_masked_stacked_TREFHT_JJA.nc")#.TREFHT.values.reshape(32 * 32, *[14307]).T
#print(predictands_pre2)

# v3 (standardized anyway)
#predictors = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v3_starting21112025/GMT_and_pseudoPCs_EOFs_Z500_5daily_100ensmembers_JJA_pcs_and_fGMT_standardized_v3.nc").pseudo_pcs.values
#print(predictors.shape)
#predictors_test = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v3_starting21112025/GMT_and_pseudoPCs_EOFs_ETH_forced_response_Z500_5daily_100ensmembers_JJA_z500_and_fgmt_standardized_v3.nc").pseudo_pcs.values
#predictands_pre2 = xr.open_dataset("/Users/friederl/Documents/EcoN_project/LLAAE/dpa_input_data/v3_starting21112025/europe_10percent_masked_stacked_TREFHT_JJA_v3.nc")#.TREFHT.values.reshape(32 * 32, *[14307]).T
#print(predictands_pre2)

(476900, 1001)
<xarray.Dataset> Size: 2GB
Dimensions:  (lat: 32, lon: 32, time: 476900)
Coordinates:
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * time     (time) object 4MB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 2GB ...


In [15]:
# Load LE temperature data
settings_file_path =  "/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder/joint_training/v4_dpa_train_settings.json" #"../joint_training/v2_dpa_train_settings.json"
    
with open(settings_file_path, 'r') as file:
        settings = json.load(file)

# Train data
trefht_le = xr.open_dataset(settings['dataset_trefht'])
print(trefht_le)

# Load LE Z500 
z500_le = xr.open_dataset(settings["dataset_z500"])
predictors_combined_le = z500_le.pseudo_pcs.values


# Load ETH ensemble data
# Test data
trefht_eth = xr.open_dataset(settings['dataset_trefht_eth_transient'])
print(trefht_eth)

z500_eth = xr.open_dataset(settings['dataset_z500_eth_test'])
print(z500_eth)
predictors_combined_eth = z500_eth.pseudo_pcs.values 
predictors_combined_eth


# germany domain 
### Germany ###
    
# coordinates 
ger_lat_min = 48
ger_lat_max = 54
ger_lon_min = 6
ger_lon_max = 15

# cut train data
trefht_le_ger = trefht_le.sel(lat=slice(ger_lat_min, ger_lat_max), lon=slice(ger_lon_min, ger_lon_max))
print(trefht_le_ger)

# cut test data
trefht_eth_ger = trefht_eth.sel(lat=slice(ger_lat_min, ger_lat_max), lon=slice(ger_lon_min, ger_lon_max))
print(trefht_eth_ger)

# calculate weighted means
#weights
weights_ger_pre = np.cos(np.deg2rad(trefht_le_ger["lat"]))
weights_ger = weights_ger_pre / weights_ger_pre.sum()

# training data
trefht_le_ger_mean = trefht_le_ger.TREFHT.weighted(weights_ger).mean(dim=("lat", "lon"))
trefht_le_ger_mean

# test_data
trefht_eth_ger_mean = trefht_eth_ger.TREFHT.weighted(weights_ger).mean(dim=("lat", "lon"))
trefht_eth_ger_mean

print("Predictors train shape:", predictors_combined_le.shape)
print("Predictands train shape:", trefht_le_ger_mean.shape)
print("#######")
print("Predictors train shape:", predictors_combined_eth.shape)
print("Predictands test shape:", trefht_eth_ger_mean.shape)


# split 

if True:
    ###
    ## train
    train_predictors, _, _ = ut.standardize_numpy(predictors_combined_le[:90*4769, :])
    X_torch = torch.from_numpy(train_predictors)
    y_torch = torch.from_numpy(trefht_le_ger_mean.values)[:90*4769].unsqueeze(1)

    ## validation
    validation_predictors, _, _ = ut.standardize_numpy(predictors_combined_le[90*4769:, :])
    X_val_torch = torch.from_numpy(validation_predictors)
    y_val_torch = torch.from_numpy(trefht_le_ger_mean.values)[90*4769:].unsqueeze(1)

    ## test
    test_predictors, _, _ = ut.standardize_numpy(predictors_combined_eth)
    X_test_torch = torch.from_numpy(test_predictors)
    y_test_torch = torch.from_numpy(trefht_eth_ger_mean.values).unsqueeze(1)

else:
    ## train
    X_torch = torch.from_numpy(predictors_combined_le)[:90*4769, :]
    y_torch = torch.from_numpy(trefht_le_ger_mean.values)[:90*4769].unsqueeze(1)

    ## validation
    X_val_torch = torch.from_numpy(predictors_combined_le)[90*4769:, :]
    y_val_torch = torch.from_numpy(trefht_eth_ger_mean.values)[90*4769:].unsqueeze(1)



print("X:", X_torch.shape)
print("y:", y_torch.shape)

print("X_val:", X_val_torch.shape)
print("y_val:", y_val_torch.shape)

print("X_test:", X_test_torch.shape)
print("y_test:", y_test_torch.shape)

<xarray.Dataset> Size: 2GB
Dimensions:  (lat: 32, lon: 32, time: 476900)
Coordinates:
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * time     (time) object 4MB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 2GB ...
<xarray.Dataset> Size: 59MB
Dimensions:  (lat: 32, lon: 32, time: 14307)
Coordinates:
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * time     (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 59MB ...
<xarray.Dataset> Size: 57MB
Dimensions:     (time: 14307, mode: 1001)
Coordinates:
  * time        (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
  * mode        (mode) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999 1000
Da

In [6]:
#predictands_pre = data_to_torch(predictands_pre2, "TREFHT")
#predictands, mask_x_tr = remove_nan_columns(predictands_pre)
#predictands.shape

torch.Size([476900, 1024])


torch.Size([476900, 648])

In [9]:
# Fit an engression model
engressor = engression(X_torch, 
                       y_torch, 
                       resblock=True, 
                       num_layer=6, 
                       hidden_dim=100, 
                       noise_dim=100, 
                       add_bn = False,
                       lr=0.001, 
                       num_epochs=30, 
                       standardize=False,
                       batch_size=128,
                       print_every_nepoch=1, 
                       device=device)

Running on CPU.

Training based on mini-batch gradient descent with a batch size of 128.
[Epoch 1 (0%), batch 3353] energy-loss: 0.6337,  E(|Y-Yhat|): 1.2726,  E(|Yhat-Yhat'|): 1.2777
[Epoch 2 (3%), batch 3353] energy-loss: 0.5390,  E(|Y-Yhat|): 1.0950,  E(|Yhat-Yhat'|): 1.1119
[Epoch 3 (7%), batch 3353] energy-loss: 0.5131,  E(|Y-Yhat|): 1.0383,  E(|Yhat-Yhat'|): 1.0504
[Epoch 4 (10%), batch 3353] energy-loss: 0.4937,  E(|Y-Yhat|): 1.0030,  E(|Yhat-Yhat'|): 1.0186
[Epoch 5 (13%), batch 3353] energy-loss: 0.4798,  E(|Y-Yhat|): 0.9712,  E(|Yhat-Yhat'|): 0.9828
[Epoch 6 (17%), batch 3353] energy-loss: 0.4687,  E(|Y-Yhat|): 0.9500,  E(|Yhat-Yhat'|): 0.9626
[Epoch 7 (20%), batch 3353] energy-loss: 0.4602,  E(|Y-Yhat|): 0.9278,  E(|Yhat-Yhat'|): 0.9353
[Epoch 8 (23%), batch 3353] energy-loss: 0.4513,  E(|Y-Yhat|): 0.9137,  E(|Yhat-Yhat'|): 0.9248
[Epoch 9 (27%), batch 3353] energy-loss: 0.4423,  E(|Y-Yhat|): 0.8962,  E(|Yhat-Yhat'|): 0.9077
[Epoch 10 (30%), batch 3353] energy-loss: 0.4360, 

In [10]:
engressor.summary()

Engression model with
	 number of layers: 6
	 hidden dimensions: 100
	 noise dimensions: 100
	 residual blocks: True
	 number of epochs: 30
	 batch size: 128
	 learning rate: 0.001
	 standardization: False
	 training mode: False
	 device: cpu

Training loss (original scale):
	 energy-loss: 0.35, 
	E(|Y-Yhat|): 0.72, 
	E(|Yhat-Yhat'|): 0.74


In [16]:
# Validation
print("L2 loss:", engressor.eval_loss(X_val_torch[:4769,:], y_val_torch[:4769,:], loss_type="l2"))
print("correlation between predicted and true means:", engressor.eval_loss(X_val_torch[:4769,:], y_val_torch[:4769,:], loss_type="cor"))
print("energy score:", engressor.eval_loss(X_val_torch[:4769,:], y_val_torch[:4769,:], loss_type="energy"))

StoNetBase.predict() Step 0
StoNetBase.sample() batch size: 4769
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
StoNetBase.predict() Step 1
L2 loss: 0.9766282138056352
StoNetBase.predict() Step 0
StoNetBase.sample() batch size: 4769
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
StoNetBase.predict() Step 1
correlation between predicted and true means: 0.9514958375168663
StoNetBase.sample() batch size: 4769
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
energy score: 0.5502391290066976


In [17]:
# Test
print("L2 loss:", engressor.eval_loss(X_test_torch, y_test_torch, loss_type="l2"))
print("correlation between predicted and true means:", engressor.eval_loss(X_test_torch, y_test_torch, loss_type="cor"))
print("energy score:", engressor.eval_loss(X_test_torch, y_test_torch, loss_type="energy"))

StoNetBase.predict() Step 0
StoNetBase.sample() batch size: 14307
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
StoNetBase.predict() Step 1
L2 loss: 1.1632800701114856
StoNetBase.predict() Step 0
StoNetBase.sample() batch size: 14307
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
StoNetBase.predict() Step 1
correlation between predicted and true means: 0.9454330074743099
StoNetBase.sample() batch size: 14307
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...
energy score: 0.6248166006319618


In [18]:
# prediction
y_pred_standardized = engressor.sample(X_test_torch, sample_size=100)
y_pred_standardized.shape

StoNetBase.sample() batch size: 14307
StoNetBase.sample_onebatch() run successfully ...
self.sample_batch() run ...


torch.Size([14307, 1, 100])

In [19]:
#torch.save(y_pred_standardized, "/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/StoNet/v4_data_1d_ger_trained_30_epochs_predictions.pt")







#